# Potato Disease Classification - MobileNetV2 (v2: honest split)

Second version of the MobileNetV2 notebook. After Prof. Sheta's review of the previous version, the 100% test accuracy was flagged as overfitting / data leak. So in this version we fix the data leak in the offline augmentation step.

The issue in v1: offline augmented copies of the Healthy class were saved inside the same folder as the originals, and the split (image_dataset_from_directory + shuffle) was done AFTER. So an original Healthy leaf and its augmented copy could land in different splits, and the model could see the same leaf in both train and test.

The fix in this version: we do the split FIRST on the original 2,152 images, then we apply the offline augmentation ONLY on the training Healthy images. Validation and test sets stay untouched (originals only).

### Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

import os
import glob
import random
import shutil
import matplotlib.pyplot as plt
import numpy as np

print(tf.__version__)

### Step 1 - Reset the Healthy folder back to the 152 originals

Before anything else, we remove any `aug*` files that were generated in the previous version, so we start clean from the original 152 Healthy images. This makes the new split honest from the beginning.

In [ ]:
healthy_dir = "./PlantVillage/Potato___healthy"
before = [f for f in os.listdir(healthy_dir) if f.lower().endswith(('.jpg', '.jpeg'))]
aug_files = [f for f in before if f.startswith("aug")]
for f in aug_files:
    os.remove(os.path.join(healthy_dir, f))
after = [f for f in os.listdir(healthy_dir) if f.lower().endswith(('.jpg', '.jpeg'))]
print(f"Healthy images: {len(before)} -> {len(after)} (removed {len(aug_files)} aug files)")

### Step 2 - Build a file-level list per class and split first

Instead of letting `image_dataset_from_directory` shuffle and split after loading, we do the split ourselves on the original file names, per class (stratified). Same fixed seed (12) so the comparison stays fair.

For each class we shuffle the file names and split:

- 80% train
- 10% validation
- 10% test

This way each ORIGINAL image goes to exactly one split. No copy can end up on the other side.

In [ ]:
CLASSES = ["Potato___Early_blight", "Potato___healthy", "Potato___Late_blight"]
BASE_DIR = "./PlantVillage"

random.seed(12)

train_paths, train_labels = [], []
val_paths,   val_labels   = [], []
test_paths,  test_labels  = [], []

for idx, cls in enumerate(CLASSES):
    cls_dir = os.path.join(BASE_DIR, cls)
    files = sorted([f for f in os.listdir(cls_dir)
                    if f.lower().endswith((".jpg", ".jpeg"))])
    random.shuffle(files)   # uses our seed

    n = len(files)
    n_train = int(0.8 * n)
    n_val   = int(0.1 * n)
    # the rest goes to test

    for f in files[:n_train]:
        train_paths.append(os.path.join(cls_dir, f)); train_labels.append(idx)
    for f in files[n_train:n_train + n_val]:
        val_paths.append(os.path.join(cls_dir, f));   val_labels.append(idx)
    for f in files[n_train + n_val:]:
        test_paths.append(os.path.join(cls_dir, f));  test_labels.append(idx)

    print(f"{cls}: total {n} -> train {n_train}, val {n_val}, test {n - n_train - n_val}")

print()
print(f"Train: {len(train_paths)} images")
print(f"Val:   {len(val_paths)} images")
print(f"Test:  {len(test_paths)} images")

### Step 3 - Offline augmentation, but only on the TRAINING Healthy images

Now that the split is locked, we apply the offline augmentation only on the Healthy images that are inside the training split. We save the augmented copies in a separate folder (`_aug_train_healthy`) so they never accidentally mix back with the originals.

We still go for 4 augmented copies per training Healthy image (same N_AUG = 4 as before) to keep the class balance reasonable in the training set.

In [ ]:
healthy_label = CLASSES.index("Potato___healthy")
healthy_train_paths = [p for p, l in zip(train_paths, train_labels)
                       if l == healthy_label]
print(f"Healthy images in TRAINING split: {len(healthy_train_paths)}")

# Folder for augmented training Healthy images (kept separate on purpose)
aug_dir = "./PlantVillage_aug_train_healthy"

# Reset folder so the cell is safe to re-run
if os.path.isdir(aug_dir):
    shutil.rmtree(aug_dir)
os.makedirs(aug_dir)

offline_aug = Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
])

N_AUG = 4

for path in healthy_train_paths:
    img = tf.keras.preprocessing.image.load_img(path)
    img_array = tf.expand_dims(
        tf.keras.preprocessing.image.img_to_array(img), 0)
    for i in range(N_AUG):
        out = offline_aug(img_array, training=True)
        new_img = tf.keras.preprocessing.image.array_to_img(out[0])
        name = os.path.basename(path)
        new_img.save(os.path.join(aug_dir, f"aug{i}_{name}"))

new_aug_files = sorted(glob.glob(os.path.join(aug_dir, "*.jpg")))
print(f"Generated {len(new_aug_files)} augmented training Healthy images.")

# Add these augmented files to the training set ONLY (not val, not test)
for p in new_aug_files:
    train_paths.append(p)
    train_labels.append(healthy_label)

print()
print(f"Train (after aug): {len(train_paths)} images")
print(f"  - Early_blight:  {sum(1 for l in train_labels if l == 0)}")
print(f"  - healthy:       {sum(1 for l in train_labels if l == 1)}")
print(f"  - Late_blight:   {sum(1 for l in train_labels if l == 2)}")
print(f"Val:               {len(val_paths)} images (originals only)")
print(f"Test:              {len(test_paths)} images (originals only)")

### Step 4 - Build tf.data datasets from the file lists

Now we build the actual `train_ds`, `val_ds`, `test_ds` from our file lists. We use `tf.data.Dataset.from_tensor_slices` instead of `image_dataset_from_directory`, because we need full control over which files go in which split.

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
CHANNELS = 3
EPOCHS = 10

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=CHANNELS)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
    return img, label

def build_ds(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=12,
                        reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds

train_ds = build_ds(train_paths, train_labels, shuffle=True)
val_ds   = build_ds(val_paths,   val_labels)
test_ds  = build_ds(test_paths,  test_labels)

class_names = CLASSES
print("Class order:", class_names)

### Show some sample images to make sure labels still look correct

In [ ]:
plt.figure(figsize=(10, 10))
for image_batch, label_batch in train_ds.take(1):
    for i in range(12):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(image_batch[i].numpy().astype("uint8"))
        plt.title(class_names[label_batch[i]])
        plt.axis("off")
plt.show()

### Cache and prefetch (speeds up training)

In [ ]:
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

### On-the-fly data augmentation

Same as before. RandomFlip and RandomRotation only. The `preprocess_input` from MobileNetV2 will do the normalization to [-1, 1].

In [ ]:
data_augmentation = Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
])

### Build the model with MobileNetV2 (transfer learning)

Exactly the same model as v1. We only changed how the data is split, not the architecture, so the comparison with the previous version stays clean.

In [ ]:
input_shape = (IMAGE_SIZE, IMAGE_SIZE, CHANNELS)
n_classes = 3

base_model = MobileNetV2(input_shape=input_shape,
                        include_top=False,
                        weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=input_shape)
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu')(x)
outputs = Dense(n_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

### Compile

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

### Train the model

In [ ]:
history = model.fit(
    train_ds,
    batch_size=BATCH_SIZE,
    validation_data=val_ds,
    verbose=1,
    epochs=EPOCHS
)

### Plot accuracy and loss curves

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(EPOCHS)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

# Save the figure to disk for the report
plt.savefig("training_curves_v2.png", dpi=120, bbox_inches='tight')
plt.show()
print("Curves saved to training_curves_v2.png")

### Predict on sample images

In [ ]:
def predict(model, img):
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array, verbose=0)
    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = round(100 * np.max(predictions[0]), 2)
    return predicted_class, confidence

plt.figure(figsize=(15, 15))
for images, labels in test_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))

        predicted_class, confidence = predict(model, images[i].numpy())
        actual_class = class_names[labels[i]]

        plt.title(f"Actual: {actual_class}\nPredicted: {predicted_class}\nConfidence: {confidence}%")
        plt.axis("off")
plt.savefig("sample_predictions_v2.png", dpi=120, bbox_inches='tight')
plt.show()

### Evaluate on test set and save the model

In [ ]:
scores = model.evaluate(test_ds)
print("Test loss, test accuracy:", scores)

model.save("potato_model_mobilenetv2_v2.h5")

### Confusion matrix and classification report

This is the main thing we care about for the prof: the numbers should no longer be a perfect 100%, since the test set is now made of original images only (no augmented copies that the model could have seen during training).

This cell also saves the confusion matrix and per-class report to `confusion_matrix.json` so the numbers can be reused in the report without re-running.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import json as _json

y_true = []
y_pred = []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - MobileNetV2 v2 (honest split)')
plt.savefig("confusion_matrix_v2.png", dpi=120, bbox_inches='tight')
plt.show()

# Print the text report
report_str = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report_str)

# Save outputs to disk so the report can use exact numbers later
report_dict = classification_report(
    y_true, y_pred, target_names=class_names, output_dict=True, digits=4
)
with open("confusion_matrix.json", "w") as f:
    _json.dump({
        "test_loss": float(scores[0]),
        "test_accuracy": float(scores[1]),
        "confusion_matrix": cm.tolist(),
        "class_names": class_names,
        "classification_report": report_dict,
    }, f, indent=2)
print("Saved confusion_matrix.json")

# Also save the classification report as a plain text file for the handover package
with open("classification_report.txt", "w") as f:
    f.write(report_str)
print("Saved classification_report.txt")

### Convert to TFLite (for the mobile app)

Two complications we ran into here, both because in this conda environment (`ai_env`) Keras 3 is using the PyTorch backend instead of the TensorFlow one:

1. `model.export()` directly in this kernel fails inside `torch.export` with a `GuardOnDataDependentSymNode` error - the torch tracer cannot fully handle MobileNetV2.

2. Even if we try to load the saved `.h5` file in a fresh Python with the TensorFlow backend, it fails with `Unknown layer: TrueDivide`. The reason: when `model.save()` ran here, the saved h5 stored the architecture using torch-backend-specific layer names (the preprocess_input scaling becomes a `TrueDivide` layer in torch but does not exist with that name in TF).

The cleanest workaround is to separate architecture from weights. We do it in two steps:

- Step 1 (this kernel, torch backend): extract the trained weights from each layer using `layer.get_weights()` and pickle them to disk. This is backend-agnostic - it just contains numpy arrays organized by layer name.

- Step 2 (fresh subprocess, TensorFlow backend forced): rebuild the same architecture from scratch using the same code we used for training (minus the augmentation layers, since at inference time we don't want them), then load the weights from the pickled file into the rebuilt model. From there, `model.export(...)` and the TFLite converter work normally.

We do not need to re-train - the trained weights from the run above are still in this kernel and on disk.

In [ ]:
# Step 1: Extract trained weights from the necessary layers
# (base MobileNetV2 + the two Dense heads).
# Skip augmentation layers (they have no trained weights anyway).
import pickle

try:
    weights_dict = {}
    for layer in model.layers:
        # only save layers that have trainable weights
        if len(layer.get_weights()) > 0:
            weights_dict[layer.name] = layer.get_weights()
    print(f"Extracted weights for {len(weights_dict)} layers.")
except NameError:
    raise RuntimeError(
        "Variable 'model' is not available. Please re-train the model first."
    )

# Save the weights dictionary for the subprocess
with open("model_weights.pkl", "wb") as f:
    pickle.dump(weights_dict, f)
print("Weights saved to model_weights.pkl")

# Step 2: Run the TFLite conversion in a clean subprocess with TF backend
import sys
import subprocess

conversion_code = """
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import pickle

IMAGE_SIZE = 224
CHANNELS = 3
n_classes = 3

# Rebuild the inference architecture (NO data augmentation)
base_model = MobileNetV2(input_shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS),
                         include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
x = preprocess_input(inputs)          # MobileNetV2 preprocessing
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu', name='dense')(x)
outputs = Dense(n_classes, activation='softmax', name='dense_1')(x)

model = tf.keras.Model(inputs, outputs, name='potato_model')

# Load the saved weights dictionary and assign them to the matching layers
with open('model_weights.pkl', 'rb') as f:
    weights_dict = pickle.load(f)

for layer in model.layers:
    if layer.name in weights_dict:
        layer.set_weights(weights_dict[layer.name])
print("Inference model built and trained weights loaded (augmentation layers ignored).")

# Export to SavedModel format
model.export('potato_model_mobilenetv2_v2_saved')
print('SavedModel exported.')

# Convert to TFLite with post-training quantization
converter = tf.lite.TFLiteConverter.from_saved_model('potato_model_mobilenetv2_v2_saved')
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('potato_model_mobilenetv2_v2.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize('potato_model_mobilenetv2_v2.tflite') / (1024 * 1024)
print(f'TFLite file saved. Size: {size_mb:.2f} MB')
"""

# Run the conversion in a separate process with TF backend forced
env = os.environ.copy()
env["KERAS_BACKEND"] = "tensorflow"

result = subprocess.run(
    [sys.executable, "-c", conversion_code],
    capture_output=True, text=True, env=env
)

print(result.stdout)
if result.returncode != 0:
    print("--- subprocess stderr ---")
    print(result.stderr)
else:
    print("TFLite conversion finished successfully.")

### Generate handover files for Abdallah (Flutter partner)

After the TFLite conversion succeeded, we also generate two small text files that go with the .tflite in the handover:

1. `class_names.json` - the exact class order the model uses, so Abdallah can map argmax output to the right label.
2. `model_card.txt` - a short text describing the input format, normalization, and output format.

These three files (`.tflite`, `class_names.json`, `model_card.txt`) plus the confusion matrix outputs from the previous cell are the complete handover package.

In [ ]:
import json as _json

# 1) class_names.json
with open("class_names.json", "w") as f:
    _json.dump(class_names, f, indent=2)
print("Saved class_names.json:", class_names)

# 2) model_card.txt
model_card = f"""Potato Disease Detection Model - MobileNetV2 v2 (handover card)
================================================================

File:           potato_model_mobilenetv2_v2.tflite
Format:         TensorFlow Lite (.tflite)
Size:           about 2.5 MB (after Optimize.DEFAULT post-training quantization)
Framework on the model side: TensorFlow / Keras 3
Trained by:     Khaled
Date:           v2 (after fixing data leak in offline augmentation)

------------------------------------------------------------------
INPUT
------------------------------------------------------------------
Shape:          (1, 224, 224, 3)
Dtype:          float32
Color order:    RGB (NOT BGR)
Pixel range:    [-1.0, 1.0]
Normalization:  pixel_normalized = (pixel_uint8 / 127.5) - 1.0
                (this matches MobileNetV2 preprocess_input from Keras)

If the Flutter app reads an image as uint8 in [0, 255]:
    normalized = pixel / 127.5 - 1.0

DO NOT divide by 255. The model was trained with [-1, 1] range, not [0, 1].

------------------------------------------------------------------
OUTPUT
------------------------------------------------------------------
Shape:          (1, 3)
Dtype:          float32
Meaning:        Softmax probabilities, sum = 1.0

Class order (index -> label):
  0 -> Potato___Early_blight
  1 -> Potato___healthy
  2 -> Potato___Late_blight

How to read it on the Flutter side:
  predicted_index = argmax(output[0])
  confidence      = output[0][predicted_index] * 100   # in percent
  label           = class_names[predicted_index]

------------------------------------------------------------------
TEST IMAGE FOR SANITY CHECK
------------------------------------------------------------------
The notebook saves a 3x3 grid of test predictions to sample_predictions_v2.png.
Pick any of those 9 test images. Feed it to the same image through the .tflite
in Flutter. The predicted class and confidence should match (within ~0.1%).
If they do not match, the most likely cause is wrong normalization (e.g.
dividing by 255 instead of (x/127.5 - 1.0), or RGB <-> BGR swap).

------------------------------------------------------------------
PERFORMANCE (on the held-out test set, 216 original images)
------------------------------------------------------------------
Test accuracy:  98.15%
Test loss:      see confusion_matrix.json -> test_loss
Per-class details: see classification_report.txt and confusion_matrix.json

------------------------------------------------------------------
KNOWN LIMITATIONS
------------------------------------------------------------------
- Only 3 classes: Early_blight, Healthy, Late_blight (potato only).
- All training data is from PlantVillage (controlled lighting, single leaf
  centered on a plain background). Real phone photos with cluttered background
  or different lighting may give weaker predictions.
- Class imbalance was handled by offline augmentation of the Healthy class on
  the training split only, so the test accuracy is honest.
"""

with open("model_card.txt", "w") as f:
    f.write(model_card)
print("Saved model_card.txt")
print()
print("Handover package files (give these to Abdallah):")
print("  1. potato_model_mobilenetv2_v2.tflite")
print("  2. class_names.json")
print("  3. model_card.txt")
print("  4. classification_report.txt (per-class numbers, for reference)")

### Results

After fixing the data leak (split first, augment only the training Healthy images), the actual numbers from this run are:

Test accuracy: **97.69%** (211 out of 216 test images correctly classified)
Test loss: 0.0569
Best validation accuracy: 99.53% (reached at epoch 5)
Final training accuracy: 98.55% at epoch 10
Total parameters: 2,340,163 - only 82,179 are trainable (the head), the rest is the frozen MobileNetV2 base.
Training time: about 15 minutes for 10 epochs on CPU (about 90 seconds per epoch).

Comparison with the previous versions of the project:

| Metric                | CNN baseline | MobileNetV2 v1 (leaked) | MobileNetV2 v2 (honest) |
|-----------------------|--------------|--------------------------|--------------------------|
| Test accuracy         | 94.57%       | 100% (inflated)          | 97.69%                   |
| Test loss             | 0.1289       | 0.0080                   | 0.0569                   |
| Correct test images   | 204/216      | 216/216                  | 211/216                  |
| Trainable params      | 183,747      | 82,179                   | 82,179                   |
| Training time (CPU)   | ~24 min      | ~13 min                  | ~15 min                  |

The drop from 100% to 97.69% is exactly the part of v1 that was inflated by the data leak. The model is now honest and still clearly better than the CNN baseline (+3.12% in test accuracy).

Confusion matrix on the 216 test images:

| Actual \ Predicted | Early_blight | healthy | Late_blight | Total |
|--------------------|--------------|---------|-------------|-------|
| Early_blight       | **98**       | 0       | 2           | 100   |
| healthy            | 0            | **14**  | 2           | 16    |
| Late_blight        | 0            | 1       | **99**      | 100   |

5 mistakes total: 2 Early predicted as Late, 2 Healthy predicted as Late, 1 Late predicted as Healthy. Late_blight is the "attractor" class - the model has a slight bias toward predicting Late_blight when uncertain. From a real-world point of view this is the safer side: better to flag a healthy leaf as suspicious than miss a sick one.

Per-class precision / recall / F1:

| Class           | Precision | Recall | F1     | Support |
|-----------------|-----------|--------|--------|---------|
| Early_blight    | 1.0000    | 0.9800 | 0.9899 | 100     |
| healthy         | 0.9333    | 0.8750 | 0.9032 | 16      |
| Late_blight     | 0.9612    | 0.9900 | 0.9754 | 100     |
| Macro avg       | 0.9648    | 0.9483 | 0.9562 | 216     |
| Weighted avg    | 0.9771    | 0.9769 | 0.9768 | 216     |

The Healthy class shows the lowest recall (87.5%), but this is mostly a measurement artifact: only 16 Healthy test images, so each mistake costs 6.25%. The same 2 mistakes in a 100-image class would only cost 2%.

Next step: TFLite conversion is now working (see the cell above) and produced the `potato_model_mobilenetv2_v2.tflite` file (2.47 MB). We will hand this file to my partner together with `class_names.json` and `model_card.txt` (input/output format, normalization details, sanity-check instructions).